# CS-781 Phase 6: Exp 2v2 — Layer-Wise (Model 3 GPT-NeoX) with IMPROVED Generation
Fixes: max_new_tokens=850, no_repeat_ngram_size=0, repetition_penalty=1.05

In [ ]:
import os, json, glob, zipfile, shutil, time, math
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig, get_linear_schedule_with_warmup
from torch.optim import AdamW
from huggingface_hub import hf_hub_download, HfApi

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)
if device == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

HF_TOKEN = 'YOUR_HF_TOKEN'
HF_DATASET_REPO = 'Exploration-Lab/CS-781-Capstone'
TARGET_REPO = 'U-241111/cs781-submissions'

# 1. Download Phase 6 Dataset
os.makedirs('/kaggle/working/data_p6', exist_ok=True)
p6_files = [
    'Phase_6_Summ/Train/train.json',
    'Phase_6_Summ/Test/test.json'
]
for f in p6_files:
    fname = os.path.basename(f)
    dest = f'/kaggle/working/data_p6/{fname}'
    if not os.path.exists(dest):
        print(f'Downloading {fname}...')
        loc = hf_hub_download(repo_id=HF_DATASET_REPO, filename=f, repo_type='dataset', token=HF_TOKEN)
        shutil.copyfile(loc, dest)

with open('/kaggle/working/data_p6/train.json') as f:
    train_full = json.load(f)
with open('/kaggle/working/data_p6/test.json') as f:
    test_data = json.load(f)

# Deterministic held-out validation split (200 samples)
import random
random.seed(42)
shuffled = list(train_full)
random.shuffle(shuffled)
val_data = shuffled[:200]
train_data = shuffled[200:]

print(f'Train samples: {len(train_data)} | Val samples: {len(val_data)} | Test samples: {len(test_data)}')


In [ ]:
# 2. Locate Checkpoint and Initialize Tokenizer
model_ckpt_dir = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'config.json' in files:
        model_ckpt_dir = root
        break
print('Model Checkpoint Path:', model_ckpt_dir, os.listdir(model_ckpt_dir))

tok = AutoTokenizer.from_pretrained(model_ckpt_dir)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token if tok.eos_token is not None else '<|endoftext|>'
tok.padding_side = 'left' # Left-pad for causal generation batching
tok.truncation_side = 'left'

def splice_document_tokens(tokens, max_prompt_tokens=896):
    """Extract Head (448) + Middle (192) + Tail (256) to ensure legal preamble, issues, and final ruling are preserved."""
    if len(tokens) <= max_prompt_tokens:
        return tokens
    head = tokens[:448]
    mid_start = max(0, len(tokens) // 2 - 96)
    mid = tokens[mid_start : mid_start + 192]
    tail = tokens[-256:]
    return head + mid + tail

class SummarizationDataset(Dataset):
    def __init__(self, data, tokenizer, max_prompt_len=896, max_summary_len=512, is_test=False):
        self.data = data
        self.tok = tokenizer
        self.max_prompt_len = max_prompt_len
        self.max_summary_len = max_summary_len
        self.is_test = is_test

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        doc_text = item.get('document') or item.get('text', '')
        raw_doc_tokens = self.tok.encode(doc_text, add_special_tokens=False)
        spliced_doc = splice_document_tokens(raw_doc_tokens, self.max_prompt_len)
        
        prompt_str = 'Document:\n' + self.tok.decode(spliced_doc) + '\n\nSummary:\n'
        prompt_ids = self.tok.encode(prompt_str, add_special_tokens=False)
        
        if self.is_test:
            # For test generation, return prompt_ids with left padding up to max_prompt_len
            pad_id = self.tok.pad_token_id if self.tok.pad_token_id is not None else 0
            pad_len = max(0, self.max_prompt_len - len(prompt_ids))
            padded_input_ids = [pad_id] * pad_len + prompt_ids[-self.max_prompt_len:]
            attn_mask = [0] * pad_len + [1] * (len(padded_input_ids) - pad_len)
            return {
                'input_ids': torch.tensor(padded_input_ids, dtype=torch.long),
                'attention_mask': torch.tensor(attn_mask, dtype=torch.long),
                'id': str(item['id']),
                'prompt_str': prompt_str
            }
        
        # Training / Validation with target summary
        summary_text = item.get('summary', '').strip()
        summary_ids = self.tok.encode(summary_text, add_special_tokens=False)[:self.max_summary_len]
        eos_id = self.tok.eos_token_id if self.tok.eos_token_id is not None else 50256
        summary_ids = summary_ids + [eos_id]
        
        full_ids = prompt_ids + summary_ids
        # Strict Loss Masking: labels = -100 on prompt, so gradient ONLY updates on summary tokens!
        labels = [-100] * len(prompt_ids) + summary_ids
        
        # Pad to fixed length 1440
        max_seq_len = self.max_prompt_len + self.max_summary_len + 32
        pad_len = max(0, max_seq_len - len(full_ids))
        pad_id = self.tok.pad_token_id if self.tok.pad_token_id is not None else 0
        padded_ids = full_ids + [pad_id] * pad_len
        attn_mask = [1] * len(full_ids) + [0] * pad_len
        padded_labels = labels + [-100] * pad_len
        
        return {
            'input_ids': torch.tensor(padded_ids[:max_seq_len], dtype=torch.long),
            'attention_mask': torch.tensor(attn_mask[:max_seq_len], dtype=torch.long),
            'labels': torch.tensor(padded_labels[:max_seq_len], dtype=torch.long),
            'id': str(item['id'])
        }


In [ ]:
# 4. Initialize Model 3 (GPT-NeoX) with Partial / Layer-Wise Tuning
model = AutoModelForCausalLM.from_pretrained(model_ckpt_dir)
model.config.pad_token_id = tok.pad_token_id

# Freeze lower layers, unfreeze top 4 transformer blocks + LM head
for name, p in model.named_parameters():
    p.requires_grad = False
    if any(f'layers.{i}.' in name for i in range(8, 12)) or 'embed_out' in name or 'final_layer_norm' in name:
        p.requires_grad = True

model = model.to(device)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'Layer-Wise Model 3: Trainable {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')

SUB_ZIP_NAME = 'submission_phase6_exp2v2_layerwise.zip'


In [ ]:
# Training Loop with Gradient Accumulation & Prompt Masking
train_dataset = SummarizationDataset(train_data, tok, max_prompt_len=896, max_summary_len=512)
train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)

optimizer = AdamW([p for p in model.parameters() if p.requires_grad], lr=3e-5, weight_decay=0.01)
epochs = 2
grad_accum = 8
total_steps = (len(train_loader) // grad_accum) * epochs
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.06*total_steps), num_training_steps=total_steps)

print(f'Starting Layer-Wise training for {epochs} epochs...')
model.train()
for epoch in range(epochs):
    total_loss = 0
    t0 = time.time()
    optimizer.zero_grad()
    for step, batch in enumerate(train_loader):
        input_ids = batch['input_ids'].to(device)
        attn_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        outputs = model(input_ids=input_ids, attention_mask=attn_mask, labels=labels)
        loss = outputs.loss / grad_accum
        loss.backward()
        
        if (step + 1) % grad_accum == 0 or (step + 1) == len(train_loader):
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
        total_loss += loss.item() * grad_accum
        if (step + 1) % 400 == 0:
            print(f'Epoch {epoch+1} | Step {step+1}/{len(train_loader)} | Running Loss: {total_loss/(step+1):.4f}')
    print(f'Epoch {epoch+1}/{epochs} Complete | Avg Loss: {total_loss/len(train_loader):.4f} | Time: {time.time()-t0:.1f}s')


In [ ]:
# 5. IMPROVED Test Inference — Full-Length Summaries with Beam Search
# KEY FIXES vs v1:
# - max_new_tokens: 450 -> 850 (gold summaries average 834 words, median 660)
# - no_repeat_ngram_size: 3 -> 0 (gold has ~150 repeated 3-grams; banning them hurts legal phrasing)
# - repetition_penalty: 1.15 -> 1.05 (mild, prevents degeneration without killing legal terms)
# - min_new_tokens: 180 -> 400 (ensure sufficient depth for legal ratio decidendi)
# - length_penalty: 1.2 -> 1.0 (neutral, let model decide natural ending)

print('Starting IMPROVED generation on 100 test cases with Beam Search (v2: full-length)...')
test_dataset = SummarizationDataset(test_data, tok, max_prompt_len=896, is_test=True)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)  # batch_size=1 for max_new_tokens=850

model.eval()
test_predictions = []
t0_inf = time.time()

with torch.no_grad():
    for batch_idx, batch in enumerate(test_loader):
        input_ids = batch['input_ids'].to(device)
        attn_mask = batch['attention_mask'].to(device)
        batch_ids = batch['id']
        
        # IMPROVED beam search decoding
        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attn_mask,
            max_new_tokens=850,
            min_new_tokens=400,
            num_beams=3,
            length_penalty=1.0,
            no_repeat_ngram_size=0,
            repetition_penalty=1.05,
            early_stopping=True,
            pad_token_id=tok.pad_token_id,
            eos_token_id=tok.eos_token_id
        )
        
        for i, out_seq in enumerate(outputs):
            in_len = input_ids[i].shape[0]
            gen_tokens = out_seq[in_len:].cpu().tolist()
            summary_text = tok.decode(gen_tokens, skip_special_tokens=True).strip()
            # Clean post-processing
            if not summary_text or len(summary_text) < 20:
                summary_text = "The appeal is dismissed and the order of the High Court is affirmed."
            test_predictions.append({
                'id': str(batch_ids[i]),
                'summary': summary_text
            })
        if (batch_idx + 1) % 10 == 0 or (batch_idx + 1) == len(test_loader):
            print(f'Generated {len(test_predictions)}/{len(test_data)} test summaries in {time.time()-t0_inf:.1f}s...')

assert len(test_predictions) == len(test_data), f'Test count mismatch: {len(test_predictions)} vs {len(test_data)}'

# Print diagnostics
word_counts = [len(p["summary"].split()) for p in test_predictions]
print(f'Test generation complete: {len(test_predictions)} summaries.')
print(f'Word counts - Min: {min(word_counts)}, Mean: {sum(word_counts)/len(word_counts):.1f}, Max: {max(word_counts)}')
print('Sample 0 preview:', test_predictions[0]['summary'][:400])

# Save prediction.json in official format
with open('/kaggle/working/prediction.json', 'w', encoding='utf-8') as f:
    json.dump(test_predictions, f, indent=2)

with zipfile.ZipFile(f'/kaggle/working/{SUB_ZIP_NAME}', 'w', zipfile.ZIP_DEFLATED) as z:
    z.write('/kaggle/working/prediction.json', arcname='prediction.json')

print(f'Created {SUB_ZIP_NAME} successfully!')

# Auto-upload to Hugging Face repository
try:
    api = HfApi(token=HF_TOKEN)
    api.upload_file(
        path_or_fileobj=f'/kaggle/working/{SUB_ZIP_NAME}',
        path_in_repo=f'phase6/{SUB_ZIP_NAME}',
        repo_id=TARGET_REPO,
        repo_type='dataset'
    )
    api.upload_file(
        path_or_fileobj='/kaggle/working/prediction.json',
        path_in_repo=f"phase6/{SUB_ZIP_NAME.replace('.zip', '_predictions.json')}",
        repo_id=TARGET_REPO,
        repo_type='dataset'
    )
    print('Uploaded all Phase 6 v2 submission artifacts to HF!')
except Exception as e:
    print('HF upload exception:', e)
